# Guided tutorial: classify a small paper corpus

**Goal:** apply one classification scheme to two short papers, inspect the passages behind each result, and count the labels. This is the smallest version of EpiLens's corpus workflow: **index → retrieve → classify → review**.

The papers are synthetic. The keyword embedder and JSON responses are scripted so this runs offline. You will see the real EpiLens workflow and output shape, but the labels here are examples, **not model predictions**.

## 1. Load the example papers

Run this notebook from the repository root or from `notebooks/`. The helper contains the small example corpus and the offline stand-ins.

In [ ]:
import sys
from pathlib import Path

if (Path.cwd() / "notebooks" / "epilens_nb.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "notebooks"))
import epilens_nb as nb

papers = nb.classification_papers()
for paper_id, paper in papers.items():
    print(paper_id, "—", paper["metadata"].title)

One paper analyzes patient data; the other develops a reusable screening method. We will ask for each paper's **main contribution**. These two teaching labels are simpler than the paper's research taxonomy.

## 2. Index and inspect evidence

The helper builds a local index and metadata store. `TinyKeywordEmbedder` ranks by word overlap, so its scores are illustrative rather than semantic similarity. Notebook 02 opens up the indexing steps.

In [ ]:
work_dir = nb.bootstrap()
strategy = "guided-tutorial"
embedder = nb.TinyKeywordEmbedder(nb.CLASSIFICATION_VOCABULARY)
index, retriever = nb.build_index(work_dir / "index", papers, embedder)
academic_db = nb.build_academic_db(papers, strategy)

print("Indexed chunks:", len(index.get_points()))
for chunk in retriever.retrieve_by_paper("main contribution method tool patient data", "paper_methods", top_k=2):
    print(f"- {chunk.section_title}: {chunk.text}")

Retrieval selects candidate passages from **one paper at a time**. Read them before trusting a label: relevant evidence can be missing or drowned out by an abstract, table, or reference.

## 3. Define one task for the corpus

A task spec fixes the categories and retrieval examples. EpiLens uses it to build a prompt and a schema for the output. The same configuration is applied to both papers.

In [ ]:
from epilens.workflows.registry import TaskSpec, build_classifier_config_from_spec

spec = TaskSpec.model_validate({
    "key": "main_contribution_demo",
    "kind": "classifier",
    "label": "Main contribution (teaching example)",
    "multi_label": False,
    "default_label": "needs_review",
    "labels": [
        {
            "code": "applied_analysis",
            "name": "Applied analysis",
            "definition": "Uses data to answer a clinical or public health question.",
            "examples": ["Patient data are analyzed to inform a hospital decision."],
        },
        {
            "code": "method_or_tool",
            "name": "Method or tool",
            "definition": "Develops or evaluates a reusable method, model, or workflow.",
            "examples": ["The paper introduces a screening tool and evaluates it."],
        },
        {
            "code": "needs_review",
            "name": "Needs review",
            "definition": "The main contribution is unclear from the available evidence.",
            "examples": [],
        },
    ],
})
config = build_classifier_config_from_spec(spec)
print(config.category_labels)

## 4. Run the classifier on both papers

`FixedJSONGenerator` plays back a response written below for each paper. It ignores the retrieved passages; a real `LLMGenerator` would use the *same* task config and retrieved evidence to produce each response. Keeping the playback visible makes it possible to learn the workflow without a provider key or hidden model behavior.

In [ ]:
from epilens.workflows import PaperClassifier

scripted_responses = {
    "paper_open_data": {
        "classification": ["applied_analysis"],
        "reasoning": "The paper analyzes patient records and discusses hospital guidance.",
    },
    "paper_methods": {
        "classification": ["method_or_tool"],
        "reasoning": "The paper introduces and evaluates a reusable screening tool.",
    },
}

runs = {}
for paper_id, response in scripted_responses.items():
    classifier = PaperClassifier(
        retriever=retriever,
        generator=nb.FixedJSONGenerator(response),
        strategy_name=strategy,
        config=config,
        academic_db=academic_db,
    )
    runs[paper_id] = classifier.run_detailed(paper_id)

for paper_id, run in runs.items():
    print(paper_id, "→", run.decision.result.classification)

## 5. Audit before counting

The schema checks that the response has allowed fields and labels. It cannot tell whether the label is true. `run_detailed` exposes candidate passages and the raw response for review. The scripted generator ignores those passages and adds no provenance. In a model run, provenance helps trace evidence but still does not prove correctness.

In [ ]:
for paper_id, run in runs.items():
    print()
    print(f"{paper_id}: {run.decision.result.classification[0]}")
    print("Reason:", run.decision.result.evidence["reasoning"])
    print("Candidate evidence:")
    for chunk in run.decision.top_evidence[:2]:
        print(f"  - {chunk.section_title}: {chunk.text[:180]}")

**Try it:** compare each scripted reason with its retrieved passages and the original sections in `papers`. Would you accept the label, or send it for human review? In a real study, define the categories first, validate on labelled examples, and audit disagreements before interpreting counts.

In [ ]:
from collections import Counter

label_counts = Counter(
    run.decision.result.classification[0]
    for run in runs.values()
)
print(dict(label_counts))

These counts only summarize two synthetic, scripted examples. A classifier produces comparable labels that can be counted **after validation**. A `PrecisionMiner` instead produces per-paper lists of candidate items for verification; see notebook 05. Notebook 02 shows the indexing and retrieval mechanics, and notebook 07 shows how to author other task specs.